In [32]:
import os
import sys
import json
import glob
import gc

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType
from snowflake.snowpark import Window

snowflake_conn_prop = Snowflake_configuration.ds1_role_json
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [33]:
# ============================================================
# 1. SKU Supercedence + Model Family Mapping
# ============================================================
def fetchSKUSupercedence_snowpark(session):
    data = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")
    data_1 = session.table("MOP_DATABASE.SOQ.MODEL_FAMILY_MAPPING")
    sku_super = data.join(data_1, on="MODEL", how="left")
    sku_super = sku_super.with_column("SKU_UNIQUE_FAMILY_CODE", F.col("UNIQUEFAMILYCODE"))
    for old_col in sku_super.columns:
        new_col = old_col.replace('"', '')
        sku_super = sku_super.rename(old_col, new_col)
    sku_super = sku_super.with_column(
        "MODEL_FAMILY_CODE",
        F.concat(
            F.col("MODEL_FAMILY"),
            F.lit('<>'),
            F.substring(
                F.col("UNIQUEFAMILYCODE"),
                F.charindex(F.lit('<>'), F.col("UNIQUEFAMILYCODE")) + F.lit(2)
            )
        )
    )
    sku_super = sku_super.rename("UNIQUEFAMILYCODE", "UNIQUE FAMILY CODE")
    return sku_super

sku_super = fetchSKUSupercedence_snowpark(session)


# ============================================================
# 2. Models for Forecasting
# ============================================================
def return_models_for_forecasting(session, use_selected_models):
    if not use_selected_models:
        return None
    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    return models_for_forecasting["MODEL_NAME"].tolist()

name_of_models = return_models_for_forecasting(session, True)


# ============================================================
# 3. ECR Sales
# ============================================================
def get_ecr_sales_snowpark(session, customer_types, start_date, end_date, name_of_models):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(F.col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((F.col("CAL_DATE") >= F.lit(start_date)) & (F.col("CAL_DATE") <= F.lit(end_date)))
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(F.col("MODEL").isin(name_of_models))
    ecr_sales = ecr_sales.with_column(
        "NET_SALES",
        F.when(
            (F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")) < 0,
            F.lit(0)
        ).otherwise(
            F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")
        )
    )
    return ecr_sales

ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-09-01',
    end_date='2026-09-24',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
ecr_sales = ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES','MODEL')


#Filter out the defunct dealers and Premia dealers
dealer_master = session.sql("""SELECT DISTINCT TRIM(DEALER_CODE) AS DEALER_CODE,TRIM(PARENT_DEALER_CODE)
                            AS PARENT_DEALER_CODE,
                            (CASE WHEN (NOT REGEXP_LIKE(DEALER_CODE,'^17.*')
                            AND LOWER(ORG_STATUS) = 'active'
                            AND ((DEALER_DFNC IS NULL) OR (DEALER_DFNC = 'Y')) ) THEN 'Y' ELSE 'N' END) AS DEALER_OF_CHOICE
                            FROM ANALYTICS_DATABASE.ANALYTICS_SALES.VW_DEALER_MASTER
                            """)

#Only keep the dealers that are there in dealer_master df
ecr_sales_valid_dealers = ecr_sales.join(dealer_master,on=["DEALER_CODE"],how="left")
ecr_sales_valid_dealers = ecr_sales_valid_dealers.filter(F.col("DEALER_OF_CHOICE")=='Y').drop("DEALER_OF_CHOICE")

In [34]:
# ecr_sales_valid_dealers.group_by("DEALER_OF_CHOICE").agg(F.sum("NET_SALES").alias("SALES_BY_DEALERS")).show()

ecr_sales_valid_dealers.show()

------------------------------------------------------------------------------------------------------------------
|"DEALER_CODE"  |"CAL_DATE"           |"SKU"           |"NET_SALES"  |"MODEL"             |"PARENT_DEALER_CODE"  |
------------------------------------------------------------------------------------------------------------------
|64781          |2026-09-19 00:00:00  |HSPLMDRSCFIRPB  |1.000000     |SPLENDOR +          |11959                 |
|24363          |2026-09-04 00:00:00  |HDESYHSZCFIGMM  |1.000000     |DESTINI 125         |10500                 |
|22342          |2026-09-02 00:00:00  |HSPPLHRSCFIBHG  |1.000000     |SPLENDOR+ XTEC 2.0  |11245                 |
|25068          |2026-09-13 00:00:00  |HSPLMIRSCFIBHG  |1.000000     |SPLENDOR +          |11365                 |
|23557          |2026-09-13 00:00:00  |HSPUNIRSCFIBLA  |1.000000     |SPLENDOR +          |12154                 |
|12058          |2026-09-10 00:00:00  |HSPLMDRSCFIRPB  |1.000000     |SPLENDOR +

In [54]:
#Actual sales by PARENT_DEALER_CODE, SKU, and CAL_DATE
ecr_agg_sales=ecr_sales_valid_dealers.group_by("PARENT_DEALER_CODE","SKU","CAL_DATE").agg(F.sum("NET_SALES").alias("TOTAL_SALES"))
ecr_agg_sales.show()


-------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"SKU"           |"CAL_DATE"           |"TOTAL_SALES"  |
-------------------------------------------------------------------------------
|11915                 |HXTRSASSCFIBGY  |2026-09-05 00:00:00  |0.000000       |
|11975                 |HSPLMTSSCFITGB  |2026-09-14 00:00:00  |1.000000       |
|10480                 |HXTRACSSCFIBLK  |2026-09-05 00:00:00  |0.000000       |
|10725                 |HXTRPSSSCFIABO  |2026-09-08 00:00:00  |1.000000       |
|11072                 |HDLHADRSCFIBKB  |2026-09-09 00:00:00  |0.000000       |
|11327                 |HDSTRHSZCFIGMM  |2026-09-21 00:00:00  |1.000000       |
|10112                 |HSPLMIRSCFIBHG  |2026-09-21 00:00:00  |1.000000       |
|12210                 |HPPLBIRSCFIBHG  |2026-09-07 00:00:00  |1.000000       |
|10199                 |HSPLMDRSCFIBHG  |2026-09-19 00:00:00  |1.000000       |
|10257                 |HSPLMDRSCFIBBK  

### Assigning A, B, and C category

In [43]:
#Aggregated sales at PARENT_DEALER_CODE and SKU level
ecr_sales_agg = ecr_sales_valid_dealers.group_by("PARENT_DEALER_CODE","SKU").agg(F.sum("NET_SALES").alias("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL"))

#Creating the partition by PARENT_DEALER_CODE
window_spec = (
    Window.partition_by("PARENT_DEALER_CODE")
    .order_by(F.col("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL").desc())
    .rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)
)
# Sum of all sales per dealer (unordered window = entire partition)
dealer_total_window = Window.partition_by("PARENT_DEALER_CODE")

new_df = ecr_sales_agg.with_column(
    "CUMULATIVE_SALES",
    F.sum("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL").over(window_spec)
)

new_df = new_df.with_column(
    "CUMULATIVE_PCT",
    F.when(
        F.sum("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL").over(dealer_total_window) == 0,
        F.lit(0)
    ).otherwise(
        F.col("CUMULATIVE_SALES") / F.sum("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL").over(dealer_total_window)
    )
)

#Assign A, B, and C category 
new_df = new_df.with_column(
    "ABC_CATEGORY",
    F.when(F.col("CUMULATIVE_PCT") <= 0.80, F.lit("A"))
     .when(F.col("CUMULATIVE_PCT") <= 0.95, F.lit("B"))
     .otherwise(F.lit("C"))
)

abc_cat = new_df.drop("CUMULATIVE_SALES")


In [44]:
abc_cat.show()

------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"SKU"           |"TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL"  |"CUMULATIVE_PCT"  |"ABC_CATEGORY"  |
------------------------------------------------------------------------------------------------------------------------
|11513                 |HSPUNIRSCFIBLA  |125.000000                                |0.181422351234    |A               |
|11513                 |HSPLMIRSCFIBHG  |118.000000                                |0.352685050798    |A               |
|11513                 |HSPPLHRSCFIBHG  |106.000000                                |0.506531204644    |A               |
|11513                 |HSPUNIRSCFIMAG  |53.000000                                 |0.583454281567    |A               |
|11513                 |HSPLMIRSCFISBK  |27.000000                                 |0.622641509434    |A               |
|11513                 |HSPLMDRS

In [51]:
total_sales = abc_cat.select(F.sum("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL").alias("GRAND_TOTAL"))

abc_summary = abc_cat.group_by("ABC_CATEGORY") \
    .agg(F.sum("TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL").alias("SALES_BY_CATEGORY"))

abc_summary = abc_summary.cross_join(total_sales)

abc_summary = abc_summary.with_column(
    "PCT_OF_TOTAL",
    F.col("SALES_BY_CATEGORY") / F.col("GRAND_TOTAL")
).drop("GRAND_TOTAL")

abc_summary.show()

---------------------------------------------------------
|"ABC_CATEGORY"  |"SALES_BY_CATEGORY"  |"PCT_OF_TOTAL"  |
---------------------------------------------------------
|A               |239048.000000        |0.784092996405  |
|C               |16120.000000         |0.052874649033  |
|B               |49704.000000         |0.163032354562  |
---------------------------------------------------------



In [55]:
abc_cat.show()

------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"SKU"           |"TOTAL_SALES_AT_PARENT_DEALER_SKU_LEVEL"  |"CUMULATIVE_PCT"  |"ABC_CATEGORY"  |
------------------------------------------------------------------------------------------------------------------------
|11513                 |HSPUNIRSCFIBLA  |125.000000                                |0.181422351234    |A               |
|11513                 |HSPLMIRSCFIBHG  |118.000000                                |0.352685050798    |A               |
|11513                 |HSPPLHRSCFIBHG  |106.000000                                |0.506531204644    |A               |
|11513                 |HSPUNIRSCFIMAG  |53.000000                                 |0.583454281567    |A               |
|11513                 |HSPLMIRSCFISBK  |27.000000                                 |0.622641509434    |A               |
|11513                 |HSPLMDRS

In [56]:
snowflake_utils.shape_of_snowpark_df(abc_cat)

(40576, 5)

In [57]:
snowflake_utils.shape_of_snowpark_df(ecr_agg_sales)

(155021, 4)

In [61]:
#Join to get A,B, and C
actual_sales_df = ecr_agg_sales.join(abc_cat,on=["PARENT_DEALER_CODE","SKU"]).select(F.col("PARENT_DEALER_CODE"),F.col("SKU"),F.col("CAL_DATE"),F.col("TOTAL_SALES").alias("NET_SALES"),F.col("ABC_CATEGORY"))
actual_sales_df.show()

----------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"SKU"           |"CAL_DATE"           |"NET_SALES"  |"ABC_CATEGORY"  |
----------------------------------------------------------------------------------------------
|11716                 |HSPPLDRSCFIBHG  |2026-09-14 00:00:00  |0.000000     |C               |
|10666                 |HSPPLHRSCFIBHG  |2026-09-03 00:00:00  |1.000000     |A               |
|10421                 |HSPUNHRSCFIIDG  |2026-09-05 00:00:00  |1.000000     |B               |
|10261                 |HPPLDISSCFIIDG  |2026-09-12 00:00:00  |1.000000     |C               |
|12205                 |HXOMKDSZCFISRD  |2026-09-11 00:00:00  |1.000000     |A               |
|11461                 |HSPLMTSSCFITGB  |2026-09-20 00:00:00  |1.000000     |B               |
|10206                 |HSPLMTRSCFITGB  |2026-09-04 00:00:00  |1.000000     |A               |
|11000                 |HDSTMDSZCFIMNB  |2026-09-0

In [52]:
from snowflake.snowpark import Window
from snowflake.snowpark import functions as F

# ============================================================
# 1. SKU Supercedence + Model Family Mapping
# ============================================================
def fetchSKUSupercedence_snowpark(session):
    data = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")
    data_1 = session.table("MOP_DATABASE.SOQ.MODEL_FAMILY_MAPPING")
    result = data.join(data_1, on="MODEL", how="left")
    result = result.with_column("SKU_UNIQUE_FAMILY_CODE", F.col("UNIQUEFAMILYCODE"))
    for old_col in result.columns:
        new_col = old_col.replace('"', '')
        result = result.rename(old_col, new_col)
    result = result.with_column(
        "MODEL_FAMILY_CODE",
        F.concat(
            F.col("MODEL_FAMILY"),
            F.lit('<>'),
            F.substring(
                F.col("UNIQUEFAMILYCODE"),
                F.charindex(F.lit('<>'), F.col("UNIQUEFAMILYCODE")) + F.lit(2)
            )
        )
    )
    result = result.rename("UNIQUEFAMILYCODE", "UNIQUE FAMILY CODE")
    return result

result = fetchSKUSupercedence_snowpark(session)

# ============================================================
# 2. Models for Forecasting
# ============================================================
def return_models_for_forecasting(session, use_selected_models):
    if not use_selected_models:
        return None
    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    return models_for_forecasting["MODEL_NAME"].tolist()

name_of_models = return_models_for_forecasting(session, True)

# ============================================================
# 3. ECR Sales
# ============================================================
def get_ecr_sales_snowpark(session, customer_types, start_date, end_date, name_of_models):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(F.col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((F.col("CAL_DATE") >= F.lit(start_date)) & (F.col("CAL_DATE") <= F.lit(end_date)))
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(F.col("MODEL").isin(name_of_models))
    ecr_sales = ecr_sales.with_column(
        "NET_SALES",
        F.when(
            (F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")) < 0,
            F.lit(0)
        ).otherwise(
            F.col("INVOICED_SALES") + F.col("CANCELLED_SALES") + F.col("RETURNED_SALES")
        )
    )
    return ecr_sales

ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-06-01',
    end_date='2026-08-31',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
ecr_sales = ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES')

# ============================================================
# 4. Active SKU Supercedence
# ============================================================
active_sku_supercedence = result.filter(F.lower(F.col("SKUSTATUS")) == 'active')
active_sku_supercedence = active_sku_supercedence.select("SKU", "MODEL_FAMILY_CODE")

# ============================================================
# 5. OBD Mapping - Map old SKUs to current active SKUs
# ============================================================
obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW")
sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")

obd_data_joined = obd_data.join(
    sku_supercedence.select("SKU", "SKUSTATUS"),
    obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"],
    how='left'
)
obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
    .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU")

ecr_sales = ecr_sales.join(
    obd_data_active_skus,
    ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"],
    how="left"
)
ecr_sales = ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

# ============================================================
# 6. Aggregate to collapse multiple transaction rows
# ============================================================
ecr_sales = ecr_sales.group_by("DEALER_CODE", "CAL_DATE", "SKU") \
    .agg(F.sum("NET_SALES").alias("NET_SALES"))

# ============================================================
# 7. Join sales with active SKU supercedence (inner join)
# ============================================================
joined_df = ecr_sales.join(right=active_sku_supercedence, on=["SKU"])
joined_df = joined_df.select("DEALER_CODE", "CAL_DATE", "MODEL_FAMILY_CODE", "SKU", "NET_SALES")

# ============================================================
# 8. Parent Dealer Code from ORG_HIERARCHY (same as training procedure)
# ============================================================
parent_map = session.table("FIVETRAN_DATABASE.ORACLE_LDP_OLAP_SCHEMA.WC_INT_ORG_DH").select(
    F.col("X_DEALER_CODE_HIER").alias("DEALER_CODE"),
    F.trim(F.split_part(F.col("PAR_ORG_NAME"), F.lit("-"), F.lit(1))).alias("PARENT_DEALER_CODE")
).distinct()

# ============================================================
# 9. Family-level daily sales (dealer grain)
# ============================================================
family_level_daily_sales = parent_map.join(joined_df, on=["DEALER_CODE"])
family_level_daily_sales = family_level_daily_sales.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.concat(F.trim(F.col("PARENT_DEALER_CODE")), F.lit('<>'), F.col("MODEL_FAMILY_CODE"))
)

# ============================================================
# 10. Aggregate at parent_dealer+family and parent_dealer+family+SKU
# ============================================================
family_sales_3_months = family_level_daily_sales \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_FAMILY_SALES"))

sku_level_sales_3_months = family_level_daily_sales \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY", "SKU") \
    .agg(F.sum("NET_SALES").alias("LAST_3_MONTHS_SKU_SALES"))

family_and_sku = family_sales_3_months.join(
    right=sku_level_sales_3_months,
    on=["PARENT_DEALER_CODE_MODEL_FAMILY"]
)

# ============================================================
# 11. SKU Proportion (using DIV0 to avoid division by zero)
# ============================================================
sku_count = family_and_sku \
    .group_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .agg(F.count("SKU").alias("SKU_COUNT"))

family_and_sku = family_and_sku.join(sku_count, on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

family_and_sku = family_and_sku.with_column(
    "PROPORTION",
    F.when(F.col("LAST_3_MONTHS_FAMILY_SALES") > 0,
           F.col("LAST_3_MONTHS_SKU_SALES") / F.col("LAST_3_MONTHS_FAMILY_SALES"))
    .when(F.col("SKU_COUNT") > 0,
           F.lit(1) / F.col("SKU_COUNT"))
    .otherwise(F.lit(0))
)

# ============================================================
# 12. ABC Classification (using DIV0 for cumulative pct)
# ============================================================
w = Window.partition_by("PARENT_DEALER_CODE_MODEL_FAMILY") \
    .order_by(F.col("LAST_3_MONTHS_SKU_SALES").desc())
family_total = Window.partition_by("PARENT_DEALER_CODE_MODEL_FAMILY")

family_and_sku = family_and_sku.with_column(
    "CUMULATIVE_PCT",
    F.call_builtin(
        "DIV0",
        F.sum("LAST_3_MONTHS_SKU_SALES").over(
            w.rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)
        ),
        F.sum("LAST_3_MONTHS_SKU_SALES").over(family_total)
    )
)

family_and_sku = family_and_sku.with_column(
    "ABC_CATEGORY",
    F.when(F.col("CUMULATIVE_PCT") <= 0.80, F.lit("A"))
     .when(F.col("CUMULATIVE_PCT") <= 0.95, F.lit("B"))
     .otherwise(F.lit("C"))
)

# ============================================================
# 13. Final output - MATERIALIZE to break the lazy DAG
# ============================================================
final_output = family_and_sku.select(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    "SKU",
    "PROPORTION",
    "ABC_CATEGORY",
    "LAST_3_MONTHS_FAMILY_SALES",
    "LAST_3_MONTHS_SKU_SALES"
)
final_output.write.mode("overwrite").save_as_table("MOP_DATABASE.SOQ.TEMP_FINAL_OUTPUT")
final_output = session.table("MOP_DATABASE.SOQ.TEMP_FINAL_OUTPUT")

# ============================================================
# 14. Actual Sales (Sep 1-22) - same parent dealer source
# ============================================================
actual_ecr_sales = get_ecr_sales_snowpark(
    session,
    start_date='2026-09-01',
    end_date='2026-09-22',
    customer_types=['Individual'],
    name_of_models=name_of_models
)
actual_ecr_sales = actual_ecr_sales.select('DEALER_CODE', 'CAL_DATE', 'SKU', 'NET_SALES')

actual_ecr_sales = actual_ecr_sales.join(
    obd_data_active_skus,
    actual_ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"],
    how="left"
)
actual_ecr_sales = actual_ecr_sales.with_column("SKU", F.coalesce(F.col("CURRENT_OBD_SKU"), F.col("SKU")))

actual_ecr_sales = actual_ecr_sales.group_by("DEALER_CODE", "CAL_DATE", "SKU") \
    .agg(F.sum("NET_SALES").alias("NET_SALES"))

actual_joined_df = actual_ecr_sales.join(right=active_sku_supercedence, on=["SKU"])
actual_joined_df = actual_joined_df.select("DEALER_CODE", "CAL_DATE", "MODEL_FAMILY_CODE", "SKU", "NET_SALES")

actual_joined_df_with_pdc = parent_map.join(actual_joined_df, on=["DEALER_CODE"])
actual_joined_df_with_pdc = actual_joined_df_with_pdc.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.concat(F.trim(F.col("PARENT_DEALER_CODE")), F.lit('<>'), F.col("MODEL_FAMILY_CODE"))
)

actual_sales_df = actual_joined_df_with_pdc.select(
    "PARENT_DEALER_CODE_MODEL_FAMILY", "SKU", "CAL_DATE", "NET_SALES"
)

# ============================================================
# 15. Join predictions with proportions for SKU disaggregation
# ============================================================
pred_itr_3 = pd.read_parquet(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#3_feature_engineering\Modelling\predictions_2026_negbin_torch_convention\20260922_153432_34f1cdf5\DA7CA8~1.PAR")
pred_itr_3_mean = pred_itr_3[["PARENT_DEALER_CODE_MODEL_FAMILY", "CAL_DATE", "PRED_MEAN"]]
pred_itr_3_sf = session.create_dataframe(pred_itr_3_mean)

pred_itr_3_sf = pred_itr_3_sf.with_column(
    "PARENT_DEALER_CODE_MODEL_FAMILY",
    F.replace(F.col("PARENT_DEALER_CODE_MODEL_FAMILY"), F.lit('_'), F.lit('<>'))
)

join_with_predictions = final_output.join(pred_itr_3_sf, on=["PARENT_DEALER_CODE_MODEL_FAMILY"])

join_with_predictions_sf = join_with_predictions.select(
    'PARENT_DEALER_CODE_MODEL_FAMILY', 'SKU', 'CAL_DATE',
    'PRED_MEAN', 'PROPORTION', 'ABC_CATEGORY'
)

join_with_predictions_sf = join_with_predictions_sf.with_column(
    "SKU_PREDICTION", F.col("PROPORTION") * F.col("PRED_MEAN")
)

# ============================================================
# 16. Verification
# ============================================================
join_with_predictions_sf.select(
    F.sum("SKU_PREDICTION").alias("TOTAL_SKU_PREDICTION")
).show()

pred_itr_3_sf.select(
    F.sum("PRED_MEAN").alias("TOTAL_FAMILY_PREDICTION")
).show()

--------------------------
|"TOTAL_SKU_PREDICTION"  |
--------------------------
|2049736.1098348985      |
--------------------------

-----------------------------
|"TOTAL_FAMILY_PREDICTION"  |
-----------------------------
|2073623.4918465866         |
-----------------------------



In [14]:
join_with_predictions_sf.show()

------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"          |"SKU"           |"CAL_DATE"           |"PRED_MEAN"          |"PROPORTION"    |"ABC_CATEGORY"  |"SKU_PREDICTION"     |
------------------------------------------------------------------------------------------------------------------------------------------------------------------
|11659<>HF DELUXE<>DRUM<>SELF<>CAST<>BLACK  |HDLHCDRSCFIBLK  |2026-09-01 00:00:00  |0.1872488483511773   |0.666666666667  |A               |0.12483256556751396  |
|11659<>HF DELUXE<>DRUM<>SELF<>CAST<>BLACK  |HDLHADRSCFIBKG  |2026-09-01 00:00:00  |0.1872488483511773   |0.333333333333  |C               |0.06241628278366336  |
|11659<>HF DELUXE<>DRUM<>SELF<>CAST<>BLACK  |HDLHCDRSCFIBLK  |2026-09-02 00:00:00  |0.2010979929244891   |0.666666666667  |A               |0.13406532861639311  |
|11659<>HF DELUXE<>DRU

In [62]:
join_with_predictions_sf = join_with_predictions_sf.with_column("PARENT_DEALER_CODE",F.split_part("PARENT_DEALER_CODE_MODEL_FAMILY",F.lit('<>'),F.lit(1)))
join_with_predictions_sf.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"        |"SKU"           |"CAL_DATE"           |"PRED_MEAN"          |"PROPORTION"    |"ABC_CATEGORY"  |"SKU_PREDICTION"      |"PARENT_DEALER_CODE"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|10479<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK  |HGLATDRSCFITBK  |2026-09-20 00:00:00  |0.0900355128653263   |0.571428571429  |A               |0.05144886449451076   |10479                 |
|10479<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK  |HGLATDRSCFIBMS  |2026-09-20 00:00:00  |0.0900355128653263   |0.428571428571  |C               |0.03858664837081554   |10479                 |
|10479<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK  |HGLATDRSCFITBK  |2026-09-21 00:0

In [64]:
pred_df = join_with_predictions_sf.select("PARENT_DEALER_CODE","SKU","CAL_DATE","SKU_PREDICTION")
pred_df.show()

--------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"SKU"           |"CAL_DATE"           |"SKU_PREDICTION"      |
--------------------------------------------------------------------------------------
|10479                 |HGLATDRSCFITBK  |2026-09-20 00:00:00  |0.05144886449451076   |
|10479                 |HGLATDRSCFIBMS  |2026-09-20 00:00:00  |0.03858664837081554   |
|10479                 |HGLATDRSCFITBK  |2026-09-21 00:00:00  |0.09379288194747215   |
|10479                 |HGLATDRSCFIBMS  |2026-09-21 00:00:00  |0.070344661460481     |
|10479                 |HGLATDRSCFITBK  |2026-09-22 00:00:00  |0.08073250051328548   |
|10479                 |HGLATDRSCFIBMS  |2026-09-22 00:00:00  |0.06054937538485814   |
|10479                 |HGLATDRSCFITBK  |2026-09-23 00:00:00  |0.0837225895878518    |
|10479                 |HGLATDRSCFIBMS  |2026-09-23 00:00:00  |0.06279194219077895   |
|10479                 |HGLATDRSCFITBK  |20

In [65]:
snowflake_utils.shape_of_snowpark_df(pred_df)

(3446928, 4)

In [66]:
snowflake_utils.shape_of_snowpark_df(actual_sales_df)

(155021, 5)

In [67]:
final_joined_df = actual_sales_df.join(pred_df,on=["PARENT_DEALER_CODE","SKU","CAL_DATE"])
final_joined_df.show()

--------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"SKU"           |"CAL_DATE"           |"NET_SALES"  |"ABC_CATEGORY"  |"SKU_PREDICTION"     |
--------------------------------------------------------------------------------------------------------------------
|10830                 |HSPSEDRSCFIMAG  |2026-09-01 00:00:00  |1.000000     |B               |0.1109399410636339   |
|10830                 |HSPSEDRSCFIMAG  |2026-09-05 00:00:00  |1.000000     |B               |0.11783217309105874  |
|10830                 |HSPSEDRSCFIMAG  |2026-09-13 00:00:00  |1.000000     |B               |0.12155554547591628  |
|12056                 |HPPLBIRSCFIBHG  |2026-09-14 00:00:00  |1.000000     |B               |2.307672696978144    |
|12056                 |HPPLBIRSCFIBHG  |2026-09-17 00:00:00  |1.000000     |B               |4.0261827660421226   |
|12056                 |HPPLBIRSCFIBHG  |2026-09-19 00:00:00  |1

In [68]:
snowflake_utils.shape_of_snowpark_df(final_joined_df)

(126507, 6)

In [69]:
final_joined_df.select(F.sum("SKU_PREDICTION").alias("SKU_PREDICTION_SUM"),F.sum("NET_SALES").alias("ACTUAL_SALES_SUM")).show()

---------------------------------------------
|"SKU_PREDICTION_SUM"  |"ACTUAL_SALES_SUM"  |
---------------------------------------------
|166198.0340936216     |279278.000000       |
---------------------------------------------

